In [0]:
# MAGIC %md
# MAGIC # Notebook 03_seguridad - Gobierno y Privacidad (Ley 1581 de 2012)
# MAGIC **Objetivo:** Clasificar datos, aplicar técnicas de anonimización y definir control de acceso por roles.

from pyspark.sql.functions import col, sha2, concat, lit, substring, date_trunc

# 1. Cargar datos de la Capa Plata
df_plata = spark.read.table("default.plata_energia_g5")

# 2. TÉCNICA DE ANONIMIZACIÓN 1: Hashing Criptográfico (SHA-256)
# Se aplica al 'CodigoSICAgente' para proteger la identidad comercial del agente sin perder la capacidad de agrupar datos.
df_seguro = df_plata.withColumn("Agente_Hash_SHA256", sha2(col("CodigoSICAgente"), 256))

# 3. TÉCNICA DE ANONIMIZACIÓN 2: Enmascaramiento Parcial (Masking)
# Se ocultan los primeros caracteres del identificador para visualizaciones en dashboards públicos.
df_seguro = df_seguro.withColumn("Agente_Enmascarado", concat(lit("***"), substring(col("CodigoSICAgente"), 3, 2)))

print("🔹 Evidencia de Anonimización y Enmascaramiento (Antes vs Después):")
display(df_seguro.select("CodigoSICAgente", "Agente_Hash_SHA256", "Agente_Enmascarado").limit(5))

🔹 Evidencia de Anonimización y Enmascaramiento (Antes vs Después):


CodigoSICAgente,Agente_Hash_SHA256,Agente_Enmascarado
NEUC,9ee5a7a5ee80c1aeb99a184f12f89285880bd86debe108fb5b5c4016b1ae2458,***UC
TENC,20a55347a15288a9de54fa70cab59e1e434c9d64a992cf0b5df967716e2e9727,***NC
CMMC,0ecd57c0c4a4e075944d57d01aedc7c6e7cd733d982f0aad4bf53678ab1a0fe9,***MC
BIAC,d5656bca6a2ef26cad3ca66a993384920396b4a4a7c2fa216da7602a9861538a,***AC
EXIC,255e4593c4326690c48e1a0b00bc15c058451dede99395e7c712badb0c238ef5,***IC


In [0]:
# 4. Control de Acceso basado en Roles (RBAC) en Unity Catalog
# Nota: En Databricks Free Edition se documenta la instrucción SQL lógica de gobierno.

instrucciones_rbac = """
-- 1. Otorgar permisos de solo lectura al equipo de analistas sobre la capa plata
GRANT SELECT ON TABLE default.plata_energia_g5 TO `rol_analistas_negocio`;

-- 2. Revocar cualquier acceso público para evitar fugas de información
REVOKE ALL PRIVILEGES ON TABLE default.plata_energia_g5 FROM `public`;

-- 3. Otorgar permisos totales a los ingenieros de datos para mantenimiento
GRANT ALL PRIVILEGES ON TABLE default.plata_energia_g5 TO `rol_ingenieros_datos`;
"""

print("🔹 Políticas de Acceso Definidas (RBAC):")
print(instrucciones_rbac)

🔹 Políticas de Acceso Definidas (RBAC):

-- 1. Otorgar permisos de solo lectura al equipo de analistas sobre la capa plata
GRANT SELECT ON TABLE default.plata_energia_g5 TO `rol_analistas_negocio`;

-- 2. Revocar cualquier acceso público para evitar fugas de información
REVOKE ALL PRIVILEGES ON TABLE default.plata_energia_g5 FROM `public`;

-- 3. Otorgar permisos totales a los ingenieros de datos para mantenimiento
GRANT ALL PRIVILEGES ON TABLE default.plata_energia_g5 TO `rol_ingenieros_datos`;

